# Serving — latency / throughput measurement

Sends N sequential requests to a **running** instance of the API and reports avg / p50 / p95 / p99 latency and throughput.

**Before running this notebook**, start the API in a separate terminal: `uvicorn serving.app:app --port 8000`

In [1]:
import statistics, time, json
import requests

URL = 'http://localhost:8000/predict'
N = 100

SAMPLE_PAYLOAD = {
    'gender': 'Female', 'SeniorCitizen': 0, 'Partner': 'Yes',
    'Dependents': 'No', 'tenure': 5, 'PhoneService': 'Yes',
    'MultipleLines': 'No', 'InternetService': 'Fiber optic',
    'OnlineSecurity': 'No', 'OnlineBackup': 'No',
    'DeviceProtection': 'No', 'TechSupport': 'No',
    'StreamingTV': 'No', 'StreamingMovies': 'No',
    'Contract': 'Month-to-month', 'PaperlessBilling': 'Yes',
    'PaymentMethod': 'Electronic check',
    'MonthlyCharges': 79.85, 'TotalCharges': '399.25',
}


## Run the load test

In [2]:
latencies_ms, errors = [], 0
t_start = time.perf_counter()

for _ in range(N):
    t0 = time.perf_counter()
    try:
        resp = requests.post(URL, json=SAMPLE_PAYLOAD, timeout=5)
        resp.raise_for_status()
    except Exception:
        errors += 1
        continue
    latencies_ms.append((time.perf_counter() - t0) * 1000)

total_wall_s = time.perf_counter() - t_start
latencies_ms.sort()

def pct(p):
    if not latencies_ms:
        return None
    idx = min(int(len(latencies_ms) * p), len(latencies_ms) - 1)
    return round(latencies_ms[idx], 2)

report = {
    'requests_sent': N, 'errors': errors, 'error_rate': round(errors / N, 4),
    'avg_latency_ms': round(statistics.mean(latencies_ms), 2) if latencies_ms else None,
    'p50_latency_ms': pct(0.50), 'p95_latency_ms': pct(0.95), 'p99_latency_ms': pct(0.99),
    'total_wall_time_s': round(total_wall_s, 2),
    'throughput_req_per_sec': round(N / total_wall_s, 2) if total_wall_s > 0 else None,
}
print(json.dumps(report, indent=2))


{
  "requests_sent": 100,
  "errors": 0,
  "error_rate": 0.0,
  "avg_latency_ms": 44.77,
  "p50_latency_ms": 42.92,
  "p95_latency_ms": 47.38,
  "p99_latency_ms": 142.19,
  "total_wall_time_s": 4.48,
  "throughput_req_per_sec": 22.34
}


## Save report

In [3]:
import os
os.makedirs('../artifacts/eval', exist_ok=True)
with open('../artifacts/eval/latency_report.json', 'w') as f:
    json.dump(report, f, indent=2)
